### Data Source:
Our World in Data (https://ourworldindata.org/grapher/electricity-mix?source=renewables&metric=share_of_generation&frequency=annual)

Citation: Ember (2026); Energy Institute Statistical Review of World Energy (2026) with major processing by Our World in Data. "Share of electricity generated by renewables." Retrieved (11.13.2026).


In [1]:
import pandas as pd
import altair as alt

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1ui2jGzkbudSQeqvokKlPMKn2afeSJTB2xPcsEe-6weY/export?format=csv")
df.head()

Mounted at /content/drive


,Entity,Code,Year,Renewables
0,ASEAN (Ember),NaN,2000,19.334143
1,ASEAN (Ember),NaN,2001,19.055025
2,ASEAN (Ember),NaN,2002,17.666613
3,ASEAN (Ember),NaN,2003,16.668121
4,ASEAN (Ember),NaN,2004,15.696422


This dataset shows the percentage of electricity generated from renewable sources(solar, wind, hydropower, bioenergy, geothermal, wave, and tidal) for countries and regions around the world by year. There are 7615 rows and 4 columns.

Entity - Country or region name - Categorical variable\
Code - Country code - Categorical variable\
Year - Year of the observation - Temporal variable\
Renewables - Percentage of electricity produced from renewable sources - Quantitative variable

### Challenges and details to be aware of
1. Regions mixed with countries: The dataset includes regional and income group totals( e.g., "ASEAN(Ember), "Europe", "High-income countries" and "World"). These have no country code (645 null values in Code) and need to be removed to avoid double counting when comparing or merging.

2. Two original sources: labels like (Ember) and (EI) show that some regions appear twice, once from each source.

3. Uneven time coverage: Although the data goes back to 1919, only a few countries have early records.

4. It's a share not an amount: A small country at 90% may produce less renewable electricity than a large country at 40%, so the values show how clean the electricity is, not how much clean power it produces.

In [3]:
# Number of rows and columns in the dataset
df.shape

(7615, 4)

In [4]:
# Display column names
df.columns

Index(['Entity', 'Code', 'Year', 'Renewables'], dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7615 entries, 0 to 7614
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Entity      7615 non-null   object 
 1   Code        6970 non-null   object 
 2   Year        7615 non-null   int64  
 3   Renewables  7615 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 238.1+ KB


In [6]:
# Check maximum and minimum values in the Renewables to know its in the reasonable range
print(df["Renewables"].min())
print(df["Renewables"].max())

0.0
100.0


In [7]:
# Check unique values in Entity
df["Entity"].unique()

array(['ASEAN (Ember)', 'Afghanistan', 'Africa', 'Africa (EI)',
       'Africa (Ember)', 'Albania', 'Algeria', 'American Samoa', 'Angola',
       'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Asia',
       'Asia (Ember)', 'Asia Pacific (EI)', 'Australia', 'Austria',
       'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados',
       'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan',
       'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'CIS (EI)', 'Cambodia', 'Cameroon', 'Canada',
       'Cape Verde', 'Cayman Islands', 'Central African Republic', 'Chad',
       'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Cook Islands',
       'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Cyprus',
       'Czechia', 'Democratic Republic of Congo', 'Denmark', 'Djibouti',
       'Dominica', 'Dominican Republic', 'EU (Ember)', 'East Timor',
       'Ecuador', 'Egypt',

In [8]:
# Check unique values in year
df["Year"].unique()

array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024, 2025, 1985, 1986, 1987, 1988, 1989, 1990, 1991,
       1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 1920, 1921, 1922,
       1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933,
       1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944,
       1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955,
       1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966,
       1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977,
       1978, 1979, 1980, 1981, 1982, 1983, 1984, 1900, 1901, 1902, 1903,
       1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914,
       1915, 1916, 1917, 1918, 1919])

In [9]:
# Check for null values
df.isnull().sum()

,0
Entity,0
Code,645
Year,0
Renewables,0


In [10]:
# Drop null values
df = df.dropna()

In [11]:
# Check for null values
df.isnull().sum()

,0
Entity,0
Code,0
Year,0
Renewables,0


In [12]:
# Check for unique country codes to find duplicates
df["Code"].unique()

array(['AFG', 'OWID_AFR', 'ALB', 'DZA', 'ASM', 'AGO', 'ATG', 'ARG', 'ARM',
       'ABW', 'OWID_ASI', 'AUS', 'AUT', 'AZE', 'BHS', 'BHR', 'BGD', 'BRB',
       'BLR', 'BEL', 'BLZ', 'BEN', 'BMU', 'BTN', 'BOL', 'BIH', 'BWA',
       'BRA', 'VGB', 'BRN', 'BGR', 'BFA', 'BDI', 'KHM', 'CMR', 'CAN',
       'CPV', 'CYM', 'CAF', 'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COG',
       'COK', 'CRI', 'CIV', 'HRV', 'CUB', 'CYP', 'CZE', 'COD', 'DNK',
       'DJI', 'DMA', 'DOM', 'TLS', 'ECU', 'EGY', 'SLV', 'GNQ', 'ERI',
       'EST', 'SWZ', 'ETH', 'OWID_EUR', 'OWID_EU27', 'FLK', 'FRO', 'FJI',
       'FIN', 'FRA', 'GUF', 'PYF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA',
       'GIB', 'GRC', 'GRL', 'GRD', 'GLP', 'GUM', 'GTM', 'GIN', 'GNB',
       'GUY', 'HTI', 'OWID_HIC', 'HND', 'HKG', 'HUN', 'ISL', 'IND', 'IDN',
       'IRN', 'IRQ', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN', 'JOR', 'KAZ',
       'KEN', 'KIR', 'OWID_KOS', 'KWT', 'KGZ', 'LAO', 'LVA', 'LBN', 'LSO',
       'LBR', 'LBY', 'LTU', 'OWID_LIC', 'OWID_LMC', 'LUX', 'MAC', 

In [13]:
# Drop duplicate entries
df.drop_duplicates(inplace=True)

In [14]:
# Create a separate dataframe for rows that contains world
world = df[df['Entity'] == 'World']
world.head()

,Entity,Code,Year,Renewables
7414,World,OWID_WRL,1900,41.224450
7415,World,OWID_WRL,1901,39.497395
7416,World,OWID_WRL,1902,38.168682
7417,World,OWID_WRL,1903,36.229310
7418,World,OWID_WRL,1904,35.182040


In [15]:
# Drop the rows that has OWID in the prefix of the code
owid = df['Code'].str.contains('OWID_')
df = df[~owid]

In [16]:
# Check for number of countries in the data
df["Entity"].nunique()

213

In [17]:
# Group by year and entity
df.groupby('Year')['Entity'].nunique()

,Entity
Year,
1920,1
1921,1
1922,1
1923,1
1924,1
...,...
2021,212
2022,212
2023,210


In [18]:
# Select top EV adopted countries and filter the dataframe
countries = ["Norway", "China", "United States", "Germany", "South Africa", "United Kingdom", "Brazil", "Thailand", "Japan", "India"]

df = df[df["Entity"].isin(countries)]
df.head()

,Entity,Code,Year,Renewables
960,Brazil,BRA,1990,94.500916
961,Brazil,BRA,1991,94.626300
962,Brazil,BRA,1992,94.420300
963,Brazil,BRA,1993,95.256330
964,Brazil,BRA,1994,95.405600


In [19]:
# Number of rows and columns after cleaning and filtering
df.shape

(465, 4)

### Patterns and relationships to explore
1. Growth in renewables over time: How fast each country's share of renewable eletricity has grown. Which countries are moving fast and which have barely changed?
2. Differences between countries: Some countries already have very high renewable shares, while others still rely heavily on fossil fuels. It would be interesting to see how these groups compare and whether the gap is closing.
3. Clean grid vs. EV adoption: The most impportant question for our group is whether countries that adopt EVs quickly also have clean electricity. An EV charged on a fossil fuel heavy grid gives a much smaller climate benefit than one charged on a renewable grid.
4. Renewables and emissions: Do countries with higher renewable shares and more EVs show a bigger drop in transport CO2 and fossil fuel use?

### Does it need to be combined with more data?
Yes. On its own, this dataset only shows how clean each country's electricity is. To answer our group's questions, it needs to be merged with the other datasets on country and year. The datasets needed are
1. EV sales share to compare EV adoption with grid cleanliness.
2. Per capita CO2 emissions from transport to see whether emissions fall as EVs and renewable grow.
3. Fossil fuel consumption per capita to see whether overall fossil fuel consumption is going down.

### Visualizations I'm interested in making
1. Line chart of renewable share over time for selected countries to show growth trends.
2. Scatter plot of Renewables in X-position and EV sales share in y-position to find countries with high EV adoption has clean grids.
3. Choropleth world map of renewable share in the latest year using a sequential scheme.

In [20]:
df.to_csv("renewables_cleaned.csv")
world.to_csv("world_cleaned.csv")

In [21]:
from google.colab import files
files.download("renewables_cleaned.csv")
files.download("world_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>